# Notebook 13 — Analog-conditioned ΔpIC50 and local SAR transfer

**Purpose:** Test whether reference-based predictions outperform global molecular regression for CYP inhibition, especially under **simulated hit expansion**. This is an independent CPU-first method, not a modification of the CheMeleon checkpoint.

**Models:** (1) direct descriptor/Morgan LightGBM, (2) similarity-weighted analog values, (3) cross-fitted residual propagation, and (4) learned directed molecular-pair ΔpIC50. A conservative **predeclared** 25% local / 75% global mixture is reported; its weight is *not selected on held-out labels*.

**Validation:** original-style five-fold Bemis–Murcko scaffold splits **and** one structure-selected, hit-expansion-like holdout. The latter uses *training anchor potency* and molecular similarity only when selecting queries, never the query labels. It is a simulation, **not** an independent test set or exact reproduction of the organizers' procurement process.

Official data: https://huggingface.co/datasets/openadmet/cyp-challenge-train-test ; source background: https://www.openadmet.org/blog/announcing-openadmets-cyp-inhibition-blind-challenge/

## 0 · Install dependencies (Colab CPU)

In [ ]:
%pip -q install "rdkit>=2023.9" "lightgbm>=4.3" "scikit-learn>=1.5" "scipy>=1.11" pandas matplotlib

## 1 · Configuration and reproducibility

First run with `FAST_MODE=True`. Use `FAST_MODE=False` for **all five** scaffold folds and a stronger hit-expansion simulation. Set `PERSISTENT_ROOT` to a Google Drive folder for restartable runs. Set `DATA_OVERRIDES` to local CSVs if downloading public data is unavailable.

In [ ]:
import io, os, json, hashlib, zipfile, platform, warnings
from pathlib import Path
from urllib.request import urlopen
import numpy as np, pandas as pd, lightgbm as lgb
from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt

SEED=2026
ISOFORMS=['CYP1A2','CYP2C9','CYP2D6','CYP3A4']
FAST_MODE=True
N_FOLDS=5
FOLDS_TO_RUN=[0] if FAST_MODE else list(range(N_FOLDS))
RUN_HIT_EXPANSION=True
RUN_BLIND_SIMILARITY_AUDIT=False  # optional 750-row test STRUCTURE audit; NO hidden labels
FP_BITS=512
SIM_BITS=2048
N_TREES=60 if FAST_MODE else 260
PAIR_TREES=55 if FAST_MODE else 180
INNER_FOLDS=2 if FAST_MODE else 3
TOP_K=4 if FAST_MODE else 8
PAIR_K=3 if FAST_MODE else 7
PAIR_MIN_SIM=0.20
LOCAL_MIN_SIM=0.25
RESIDUAL_SCALE=0.35
LOCAL_BLEND_WEIGHT=0.25  # fixed *before* outer evaluation
HIT_ANCHORS_PER_CYP=4 if FAST_MODE else 25
HIT_NEIGHBORS_PER_ANCHOR=4 if FAST_MODE else 10
HIT_MIN_SIM=0.25  # report the observed distances; sparse analogs may be absent
DATA_OVERRIDES={}  # {'direct':'/content/direct.csv', 'blind':'/content/blind.csv'}
PERSISTENT_ROOT=None  # '/content/drive/MyDrive/CYP/Notebook13'
EXPERIMENT_TAG='analog_delta_v1'
ROOT_URL='https://huggingface.co/datasets/openadmet/cyp-challenge-train-test/resolve/main/'
FILES={'direct':'cyp-challenge-TRAIN_inhibition.csv','blind':'cyp-challenge-TEST-BLINDED.csv'}
BASE=Path(PERSISTENT_ROOT) if PERSISTENT_ROOT else Path('.')
OUT=BASE/'cyp_notebook_13_results';OUT.mkdir(parents=True,exist_ok=True)
CACHE=BASE/'cyp_notebook_13_cache';CACHE.mkdir(parents=True,exist_ok=True)
np.random.seed(SEED)
print('RDKit',rdBase.rdkitVersion,'LightGBM',lgb.__version__,'FAST_MODE',FAST_MODE)

def read_source(which):
    blob=Path(DATA_OVERRIDES[which]).read_bytes() if which in DATA_OVERRIDES else urlopen(ROOT_URL+FILES[which],timeout=180).read()
    return pd.read_csv(io.BytesIO(blob)),hashlib.sha256(blob).hexdigest()

def ident(s):
    m=Chem.MolFromSmiles(str(s)) if pd.notna(s) else None
    if m is None:return None,None
    can=Chem.MolToSmiles(m,canonical=True,isomericSmiles=True)
    murcko=MurckoScaffold.GetScaffoldForMol(m)
    sc=Chem.MolToSmiles(murcko,canonical=True,isomericSmiles=False) if murcko.GetNumAtoms() else 'ACYCLIC_'+can
    return can,sc

def interval_excess(pred,lo,hi):
    return np.maximum(np.asarray(lo)-pred,0)+np.maximum(pred-np.asarray(hi),0)

def save_archive():
    target=BASE/'cyp_notebook_13_results.zip'
    with zipfile.ZipFile(target,'w',zipfile.ZIP_DEFLATED) as z:
        for file in sorted(OUT.rglob('*')):
            if file.is_file():z.write(file,OUT.name+'/'+str(file.relative_to(OUT)))
    return target

## 2 · Data, fixed scaffold folds, descriptor and fingerprint features

We preserve canonical *stereochemical* SMILES for compound identity. Each scaffold group is built from the Bemis–Murcko core, with distinct acyclic molecules grouped separately. All preprocessing is structural and never uses potency labels.

In [ ]:
raw,sha_direct=read_source('direct')
required={'SMILES'}|{f'{c}_pIC50_direct_inhibition' for c in ISOFORMS}
assert required.issubset(raw.columns), sorted(required-set(raw.columns))
primary=raw.copy().reset_index(drop=True)
identity=[ident(s) for s in primary.SMILES]
primary['canonical_smiles']=[t[0] for t in identity]
primary['scaffold_group']=[t[1] for t in identity]
assert primary.canonical_smiles.notna().all() and primary.canonical_smiles.is_unique
N=len(primary)
Y=np.column_stack([pd.to_numeric(primary[f'{iso}_pIC50_direct_inhibition'],errors='coerce').to_numpy(float) for iso in ISOFORMS])
LOW=np.column_stack([pd.to_numeric(primary.get(f'{iso}_pIC50_direct_inhibition_conf_low',pd.Series(np.nan,index=primary.index)),errors='coerce').to_numpy(float) for iso in ISOFORMS])
HIGH=np.column_stack([pd.to_numeric(primary.get(f'{iso}_pIC50_direct_inhibition_conf_high',pd.Series(np.nan,index=primary.index)),errors='coerce').to_numpy(float) for iso in ISOFORMS])
scaffolds=primary.scaffold_group.to_numpy(str)
fold_assign=np.full(N,-1,int)
for f,(_,va) in enumerate(GroupKFold(n_splits=N_FOLDS).split(np.arange(N),groups=scaffolds)):
    fold_assign[va]=f
assert (fold_assign>=0).all()
assert not pd.DataFrame({'scaffold':scaffolds,'fold':fold_assign}).groupby('scaffold').fold.nunique().gt(1).any()
primary['fold']=fold_assign
primary[['canonical_smiles','scaffold_group','fold']].to_csv(OUT/'fold_manifest.csv',index=False,float_format='%.12g')

mols=[Chem.MolFromSmiles(s) for s in primary.canonical_smiles]
DESC_NAMES=['MolWt','MolLogP','TPSA','NumHDonors','NumHAcceptors','NumRotatableBonds',
            'RingCount','NumAromaticRings','FractionCSP3','HeavyAtomCount','NumHeteroatoms',
            'NHOHCount','NOCount','NumAliphaticRings','NumSaturatedRings','LabuteASA']
D=np.asarray([[getattr(Descriptors,k)(m) for k in DESC_NAMES] for m in mols],np.float32)
D[~np.isfinite(D)]=np.nan
count_gen=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=FP_BITS)
sim_gen=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=SIM_BITS)
F=np.zeros((N,FP_BITS),np.float32)
for i,m in enumerate(mols):DataStructs.ConvertToNumpyArray(count_gen.GetCountFingerprint(m),F[i])
X=np.concatenate([D,F],axis=1)
XCOL=[f'd_{n}' for n in DESC_NAMES]+[f'm_{i}' for i in range(FP_BITS)]
SIM_FPS=[sim_gen.GetFingerprint(m) for m in mols]
print('Compounds',N,'measured isoform labels',np.isfinite(Y).sum(axis=0),'molecular features',X.shape[1])

## 3 · Leakage-safe local-SAR model definitions

**Analog retrieval** uses only molecules in the *outer training partition* with that CYP measured. **Residual propagation** uses inner-scaffold-OOF training residuals, never training-fit residuals. **ΔpIC50 pairs** are constructed only among the measured training compounds and their training analogs. No held-out potency, confidence interval, or screening measurement enters fitting or retrieval.

In [ ]:
def make_lgb(seed,trees=N_TREES):
    return lgb.LGBMRegressor(n_estimators=trees,learning_rate=.045,num_leaves=15,
        min_child_samples=8,reg_lambda=5.,verbosity=-1,n_jobs=2,random_state=int(seed))

def features_for(indices):
    return pd.DataFrame(X[np.asarray(indices,int)],columns=XCOL)

def predict_similarity(query_idx,ref_idx):
    # Pairwise Tanimoto (not model cosine similarity); tie/order stable.
    ref_idx=np.asarray(ref_idx,int)
    if len(ref_idx)==0:return np.empty((len(query_idx),0))
    ref_fps=[SIM_FPS[k] for k in ref_idx]
    return np.asarray([DataStructs.BulkTanimotoSimilarity(SIM_FPS[int(i)],ref_fps) for i in query_idx],np.float32)

def neighbors(query_idx,ref_idx,k,min_sim=LOCAL_MIN_SIM,exclude_self=False):
    query_idx=np.asarray(query_idx,int);ref_idx=np.asarray(ref_idx,int)
    similarities=predict_similarity(query_idx,ref_idx)
    result=[]
    for pos,query in enumerate(query_idx):
        sim=similarities[pos].copy()
        if exclude_self: sim[ref_idx==query]=-1.
        top=np.argsort(-sim,kind='stable')[:k]
        top=top[sim[top]>=min_sim]
        result.append((ref_idx[top],sim[top]))
    return result

def inner_oof_residuals(train_idx,j,seed):
    labeled=np.asarray(train_idx,int)[np.isfinite(Y[train_idx,j])]
    if len(labeled)<12 or len(set(scaffolds[labeled]))<2:return {}
    pred=np.full(len(labeled),np.nan)
    splits=min(INNER_FOLDS,len(set(scaffolds[labeled])))
    for fold,(tr,te) in enumerate(GroupKFold(n_splits=splits).split(labeled,groups=scaffolds[labeled])):
        a,b=labeled[tr],labeled[te]
        assert not (set(scaffolds[a])&set(scaffolds[b]))
        mdl=make_lgb(seed+17+fold)
        mdl.fit(features_for(a),Y[a,j]);pred[te]=mdl.predict(features_for(b))
    assert np.isfinite(pred).all()
    return dict(zip(labeled.tolist(),(Y[labeled,j]-pred).tolist()))

def pair_features(q,a,s):
    # Directed q-a features: descriptor change, count-fingerprint change, similarity,
    # plus the anchor's structure descriptors (but never the held-out query label).
    return np.r_[np.nan_to_num(D[q]-D[a],nan=0),F[q]-F[a],
                 np.nan_to_num(D[a],nan=0),float(s)].astype(np.float32)

PAIR_COLS=([f'delta_{n}' for n in DESC_NAMES] + [f'morgan_delta_{i}' for i in range(FP_BITS)] +
           [f'anchor_{n}' for n in DESC_NAMES] + ['tanimoto'])

def fit_pair_model(labeled,j,seed):
    pairs=neighbors(labeled,labeled,PAIR_K,min_sim=PAIR_MIN_SIM,exclude_self=True)
    features=[];deltas=[];sample_w=[]
    for q,(a_idx,sims) in zip(labeled,pairs):
        for a,s in zip(a_idx,sims):
            features.append(pair_features(int(q),int(a),s))
            deltas.append(float(Y[q,j]-Y[a,j]));sample_w.append(max(.1,float(s))**2)
    if len(deltas)<25 or np.std(deltas)<1e-9:return None,len(deltas)
    model=make_lgb(seed+47,trees=PAIR_TREES)
    model.fit(pd.DataFrame(np.asarray(features),columns=PAIR_COLS),np.asarray(deltas),sample_weight=sample_w)
    return model,len(deltas)

def fit_and_predict(tr_idx,va_idx,j,seed):
    labeled=np.asarray(tr_idx,int)[np.isfinite(Y[tr_idx,j])]
    if len(labeled)<12:
        return [],{'train_labels':len(labeled),'pair_rows':0}
    assert not set(labeled).intersection(set(va_idx))
    baseline=make_lgb(seed)
    baseline.fit(features_for(labeled),Y[labeled,j])
    base=baseline.predict(features_for(va_idx))
    residuals=inner_oof_residuals(tr_idx,j,seed)
    pair_model,n_pairs=fit_pair_model(labeled,j,seed)
    close=neighbors(va_idx,labeled,TOP_K)
    rows=[]
    for k,(q,(ref,sim)) in enumerate(zip(va_idx,close)):
        base_y=float(base[k])
        if len(ref):
            weights=np.maximum(sim,1e-4)**4;weights/=weights.sum()
            analog=float(np.dot(weights,Y[ref,j]))
            # Cross-fitted residuals; absence means no correction, never fit residuals in-sample.
            resid=np.asarray([residuals.get(int(a),0.) for a in ref])
            n_resid=int(sum(int(a) in residuals for a in ref))
            correction=float(RESIDUAL_SCALE*np.dot(weights,resid))
            local=float(base_y+correction)
            if pair_model is not None:
                qf=np.stack([pair_features(int(q),int(a),float(s)) for a,s in zip(ref,sim)])
                pair_vals=Y[ref,j]+pair_model.predict(pd.DataFrame(qf,columns=PAIR_COLS))
                pair_pred=float(np.dot(weights,pair_vals))
            else:pair_pred=analog
            best=float(sim[0]);mean_sim=float(np.dot(weights,sim))
            anchor=int(ref[0]); cliff=float(np.std(Y[ref,j])) if len(ref)>1 else 0.
            # Fixed rules, selected without seeing held-out labels.
            gate=min(1.,max(0.,(best-LOCAL_MIN_SIM)/.40))
            gate*=1./(1.+cliff)
            local_blend=base_y+(LOCAL_BLEND_WEIGHT*gate)*(pair_pred-base_y)
        else:
            analog=local=pair_pred=local_blend=base_y
            best=mean_sim=0.;anchor=-1;gate=0.;n_resid=0;cliff=np.nan
        rows.append(dict(row_index=int(q),isoform=ISOFORMS[j],y_true=float(Y[q,j]),
            ci_low=LOW[q,j],ci_high=HIGH[q,j],scaffold_group=scaffolds[q],
            max_similarity=best,weighted_similarity=mean_sim,nearest_anchor=anchor,
            local_gate=gate,neighborhood_potency_std=cliff,n_oof_residuals=n_resid,
            p_direct=base_y,p_analog=analog,p_residual=local,p_delta=pair_pred,p_blend=local_blend))
    return rows,{'train_labels':len(labeled),'pair_rows':n_pairs,'crossfit_residuals':len(residuals)}

## 4 · Two validation protocols

**Scaffold:** five held-out scaffold folds, same GroupKFold layout as earlier notebooks.

**Hit expansion:** pick high-potency anchor compounds, leave the anchors in training, and withhold nearby structures selected by *SMILES similarity alone*. This resembles follow-up analog procurement; scaffolds can intentionally overlap. Candidate molecules are selected without looking at their CYP labels. We report how many actual measured responses the simulation yields, instead of assuming dense labels.

In [ ]:
def construct_hit_expansion():
    # Pre-partition molecular identities independent of ALL potency measurements.
    # Only the anchor reservoir may contribute observed potency to hit selection;
    # held-out analog candidates come from the untouched, structure-only candidate pool.
    rng=np.random.default_rng(SEED+1300)
    random_order=rng.permutation(N)
    cut=max(1,int(.75*N))
    reservoir=set(map(int,random_order[:cut]))
    candidates=set(map(int,random_order[cut:]))
    anchor_indices=[];used=set()
    for j in (0,1,3):
        eligible=np.asarray(sorted(k for k in reservoir if np.isfinite(Y[k,j])),int)
        ordered=eligible[np.argsort(-Y[eligible,j],kind='stable')]
        taken=0
        for a in ordered:
            if int(a) in used:continue
            used.add(int(a));anchor_indices.append(int(a));taken+=1
            if taken>=HIT_ANCHORS_PER_CYP:break
    anchor_set=set(anchor_indices)
    selected=set();selection_records=[]
    for anchor in anchor_indices:
        eligible=np.asarray(sorted(candidates-selected),int)
        if not len(eligible):break
        ss=predict_similarity(eligible,[anchor]).ravel()
        order=np.argsort(-ss,kind='stable')
        found=0
        for pos in order:
            q=int(eligible[pos]);sim=float(ss[pos])
            if sim<HIT_MIN_SIM:break
            selected.add(q)
            selection_records.append({'query_index':q,'anchor_index':anchor,
                                      'similarity_to_anchor':sim,
                                      'anchor_scaffold':scaffolds[anchor],
                                      'query_scaffold':scaffolds[q]})
            found+=1
            if found>=HIT_NEIGHBORS_PER_ANCHOR:break
    val=np.asarray(sorted(selected),int)
    train=np.asarray(sorted(set(range(N))-selected),int)
    assert set(anchor_indices).issubset(set(train)) and not set(train).intersection(set(val))
    sim_selection=pd.DataFrame(selection_records)
    sim_selection.to_csv(OUT/'hit_expansion_selected_analogs.csv',index=False,float_format='%.12g')
    return train,val,len(anchor_indices)

HIT_TRAIN,HIT_VAL,HIT_ANCHORS=construct_hit_expansion() if RUN_HIT_EXPANSION else (np.array([],int),np.array([],int),0)
print('Simulated hit expansion: anchors',HIT_ANCHORS,'withheld analog structures',len(HIT_VAL),
      'observed CYP labels in holdout',np.isfinite(Y[HIT_VAL]).sum(axis=0) if len(HIT_VAL) else [])

## 5 · Train, cache, and evaluate each experiment

All cached objects include the source data hash, source/feature versions, settings and fold indices. A cached prediction is rejected if its rows or experimental mode do not match. A missing isoform label is never substituted with inactivity.

In [ ]:
SETTINGS={'data_sha':sha_direct,'tag':EXPERIMENT_TAG,'seed':SEED,'fast':FAST_MODE,
    'bits':FP_BITS,'sim_bits':SIM_BITS,'n_trees':N_TREES,'pair_trees':PAIR_TREES,
    'inner_folds':INNER_FOLDS,'top_k':TOP_K,'pair_k':PAIR_K,'pair_min_sim':PAIR_MIN_SIM,
    'local_min_sim':LOCAL_MIN_SIM,'residual_scale':RESIDUAL_SCALE,'blend':LOCAL_BLEND_WEIGHT,
    'hit_anchors':HIT_ANCHORS_PER_CYP,'hit_neigh':HIT_NEIGHBORS_PER_ANCHOR,
    'hit_min_sim':HIT_MIN_SIM,'lgb_version':lgb.__version__,'rdkit':rdBase.rdkitVersion}
RUN_ID=hashlib.sha256(json.dumps(SETTINGS,sort_keys=True).encode()).hexdigest()[:16]
RUN_CACHE=CACHE/RUN_ID;RUN_CACHE.mkdir(exist_ok=True,parents=True)

def execute_split(name,fold,tr,va,scaffold_disjoint):
    tr=np.asarray(tr,int);va=np.asarray(va,int)
    assert len(set(tr)&set(va))==0
    shared=set(scaffolds[tr])&set(scaffolds[va])
    if scaffold_disjoint: assert not shared,'Scaffold leakage in purported scaffold split.'
    path=RUN_CACHE/f'{name}_fold_{fold}_predictions.csv'
    audit_path=RUN_CACHE/f'{name}_fold_{fold}_audit.csv'
    expected={(int(k),ISOFORMS[j]) for k in va for j in range(4) if np.isfinite(Y[k,j])}
    if path.exists() and audit_path.exists():
        old=pd.read_csv(path);a=pd.read_csv(audit_path)
        assert {(int(q),str(i)) for q,i in zip(old.row_index,old.isoform)}==expected
        print('RESUMED',name,fold,len(old),'predictions')
        return old,a
    rows=[];audit=[]
    for j,iso in enumerate(ISOFORMS):
        observed_val=va[np.isfinite(Y[va,j])]
        if not len(observed_val):continue
        preds,meta=fit_and_predict(tr,observed_val,j,SEED+100*fold+11*j)
        for x in preds:x.update(split_name=name,fold=int(fold),canonical_smiles=primary.canonical_smiles.iloc[x['row_index']])
        rows.extend(preds)
        audit.append(dict(split_name=name,fold=fold,isoform=iso,n_train=len(tr),n_test=len(va),
            n_test_measured=len(observed_val),scaffold_overlap=len(shared),
            molecule_overlap=len(set(tr)&set(va)),**meta))
    out=pd.DataFrame(rows);audit=pd.DataFrame(audit)
    if not out.empty:assert {(int(q),str(i)) for q,i in zip(out.row_index,out.isoform)}==expected
    out.to_csv(path,index=False,float_format='%.12g');audit.to_csv(audit_path,index=False,float_format='%.12g')
    print('FIT',name,fold,'predictions',len(out),'train analog-pair counts',audit.pair_rows.sum() if len(audit) else 0)
    return out,audit

all_pred=[];all_audit=[]
for fold in FOLDS_TO_RUN:
    tr=np.flatnonzero(fold_assign!=fold);va=np.flatnonzero(fold_assign==fold)
    p,a=execute_split('scaffold',fold,tr,va,True)
    all_pred.append(p);all_audit.append(a)
if RUN_HIT_EXPANSION and len(HIT_VAL):
    p,a=execute_split('hit_expansion',0,HIT_TRAIN,HIT_VAL,False)
    all_pred.append(p);all_audit.append(a)
PRED=pd.concat(all_pred,ignore_index=True) if all_pred else pd.DataFrame()
AUDIT=pd.concat(all_audit,ignore_index=True) if all_audit else pd.DataFrame()
PRED.to_csv(OUT/'predictions.csv',index=False,float_format='%.12g')
AUDIT.to_csv(OUT/'leakage_and_training_audit.csv',index=False,float_format='%.12g')
assert (AUDIT.molecule_overlap==0).all()
assert (AUDIT.query('split_name == "scaffold"').scaffold_overlap==0).all()

## 6 · Compare direct, analog, residual, ΔpIC50, and fixed local blend

Results are stratified by **validation scheme**, isoform, similarity to training analogs and potency. The hit-expansion cohort is not independent of the original training dataset; its results are secondary and should be interpreted as a simulated deployment scenario.

In [ ]:
MODELS=['p_direct','p_analog','p_residual','p_delta','p_blend']
metrics=[]
for (split,iso),part in PRED.groupby(['split_name','isoform']):
    y=part.y_true.to_numpy(float)
    for model in MODELS:
        pred=part[model].to_numpy(float)
        row={'split':split,'isoform':iso,'model':model,'n':len(part),
             'mae':mean_absolute_error(y,pred),
             'bias':float(np.mean(pred-y)),
             'pred_std_ratio':float(np.std(pred)/(np.std(y)+1e-12)),
             'r2':r2_score(y,pred) if len(y)>2 else np.nan}
        valid=np.isfinite(part.ci_low)&np.isfinite(part.ci_high)
        row['mean_interval_excess']=float(np.mean(interval_excess(pred[valid],part.ci_low[valid],part.ci_high[valid]))) if valid.any() else np.nan
        metrics.append(row)
METRICS=pd.DataFrame(metrics)
METRICS.to_csv(OUT/'model_metrics_by_isoform.csv',index=False,float_format='%.12g')
macro=METRICS.groupby(['split','model'],as_index=False).agg(macro_mae=('mae','mean'),
    macro_interval_excess=('mean_interval_excess','mean'),isoforms=('isoform','nunique'))
macro.to_csv(OUT/'macro_comparison.csv',index=False,float_format='%.12g')
print(macro.sort_values(['split','macro_mae']).to_string(index=False))

# Matched errors across structural neighborhoods. Similarity thresholds are descriptors, not selection knobs.
part=PRED.copy()
part['similarity_bucket']=pd.cut(part.max_similarity,[-.01,.25,.4,.6,.8,1.001],
    labels=['<0.25','0.25–0.40','0.40–0.60','0.60–0.80','≥0.80'])
part['direct_abs_error']=(part.y_true-part.p_direct).abs()
part['delta_abs_error']=(part.y_true-part.p_delta).abs()
part['blend_abs_error']=(part.y_true-part.p_blend).abs()
sim_diag=part.groupby(['split_name','isoform','similarity_bucket'],observed=True).agg(
    n=('row_index','size'),direct_mae=('direct_abs_error','mean'),
    delta_mae=('delta_abs_error','mean'),blend_mae=('blend_abs_error','mean')).reset_index()
sim_diag.to_csv(OUT/'performance_by_analog_similarity.csv',index=False,float_format='%.12g')
if len(macro):
    fig,ax=plt.subplots(figsize=(9,4))
    for mode,sub in macro.groupby('split'):
        ax.plot(sub.model,sub.macro_mae,marker='o',label=mode)
    ax.set(ylabel='Macro MAE (pIC50)',xlabel='Model',title='Local SAR transfer versus direct QSAR')
    ax.tick_params(axis='x',rotation=20);ax.legend();fig.tight_layout()
    fig.savefig(OUT/'local_sar_comparison.png',dpi=140);plt.show()

## 7 · Optional blind-test analog proximity audit (no hidden labels)

The public blind set is available only as structures. This diagnostic may indicate whether an analog-based method is within its applicability domain; **it cannot measure blind-test accuracy or justify tuning on test labels**.

In [ ]:
if RUN_BLIND_SIMILARITY_AUDIT:
    blind,sha_blind=read_source('blind')
    assert 'SMILES' in blind.columns
    rows=[]
    for z,smi in enumerate(blind.SMILES):
        m=Chem.MolFromSmiles(str(smi))
        if m is None:raise ValueError(f'Invalid blinded SMILES row {z}')
        fp=sim_gen.GetFingerprint(m)
        rec={'test_row':z,'Molecule_Name':blind.Molecule_Name.iloc[z] if 'Molecule_Name' in blind else z}
        for j,iso in enumerate(ISOFORMS):
            train_idx=np.flatnonzero(np.isfinite(Y[:,j]))
            ss=DataStructs.BulkTanimotoSimilarity(fp,[SIM_FPS[k] for k in train_idx])
            pos=int(np.argmax(ss));rec[f'{iso}_max_train_similarity']=float(ss[pos])
            rec[f'{iso}_nearest_known_id']=str(primary.Molecule_Name.iloc[train_idx[pos]]) if 'Molecule_Name' in primary else str(train_idx[pos])
        rows.append(rec)
    pd.DataFrame(rows).to_csv(OUT/'blind_test_similarity_ONLY_no_labels.csv',index=False,float_format='%.12g')
    print('Blind-test structural proximity audit completed; NO blind activities evaluated.')

## 8 · Export, provenance, restartability and interpretation

- A positive result must be supported by **matched held-out predictions**, not only close-analog examples.
- Scaffold and hit-expansion validation answer different questions: scaffold splits prohibit shared Bemis–Murcko cores; hit expansion intentionally retains measured hits near held-out analogs.
- Local corrections fall back to the global model when no sufficiently similar reference exists.
- Anchor selection uses potency *only in a predetermined 75% anchor reservoir*. Candidates are held out of anchor selection before any potency is inspected; query selection uses only chemical similarity.
- Training anchors are included among the fitting rows; held-out analog outcomes never inform anchor selection or model fitting.
- Earlier ChemMeleon/screening 50:50 ensemble remains the comparison target for future experiments; this notebook's global baseline is independent and computationally cheap.

In [ ]:
metadata=dict(notebook='13_analog_delta_local_sar.ipynb',run_id=RUN_ID,
    settings=SETTINGS,selected_folds=FOLDS_TO_RUN,complete_scaffold=(set(FOLDS_TO_RUN)==set(range(N_FOLDS))),
    n_structures=N,n_hit_analog_queries=len(HIT_VAL),sha_direct=sha_direct,
    methods=MODELS,source='OpenADMET public direct-inhibition training data',
    evaluation='PUBLIC_TRAIN_CV_AND_SIMULATED_HIT_EXPANSION_NOT_BLIND_TEST',
    python=platform.python_version())
(OUT/'run_metadata.json').write_text(json.dumps(metadata,indent=2,default=str))
archive=save_archive()
print('Saved:',archive.resolve())
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    pass